# CRISP-DM Machine Learning Project: Predicting Breast Cancer Diagnosis

This notebook follows the CRISP-DM process to explore, clean, model, evaluate, and interpret a supervised machine learning classification problem using the Breast Cancer Wisconsin diagnostic dataset from `scikit-learn`.

**Important note:** This is an educational project and is not intended for clinical decision-making.

## 1. Business Understanding

**Objective:** Build a machine learning model that predicts whether a tumor is malignant or benign based on diagnostic measurements.

**Questions of interest:**
1. Which diagnostic features show strong differences between malignant and benign tumors?
2. Does the dataset need cleaning?
3. Which model performs better for prediction?
4. What does a prediction look like for a new scenario?

In this context, recall is especially important because it measures how many malignant cases the model correctly identifies.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42


ModuleNotFoundError: No module named 'pandas'

## 2. Data Understanding

Load the dataset and convert it into a pandas DataFrame for easier exploration.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# In sklearn's breast cancer dataset: 0 = malignant, 1 = benign
df = X.copy()
df['diagnosis'] = y.map({0: 'malignant', 1: 'benign'})

df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('
Class distribution:')
print(df['diagnosis'].value_counts())

df.describe().T.head(10)

### Exploratory Data Analysis: Class Balance

The following chart shows the distribution of benign and malignant cases.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='diagnosis')
plt.title('Distribution of Diagnosis Classes')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.show()

### Distribution Plots and Skewness

Distribution plots help us understand whether variables are symmetric or skewed. Skewed variables may influence certain models and may require transformation depending on the modeling approach.

In [ ]:
selected_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']

for feature in selected_features:
    plt.figure(figsize=(7,4))
    sns.histplot(data=df, x=feature, hue='diagnosis', kde=True, bins=30)
    plt.title(f'Distribution of {feature}')
    plt.show()

In [ ]:
skewness = X.skew().sort_values(ascending=False)
skewness.head(10)

### Correlation Analysis

A correlation heatmap helps identify highly related variables. Many diagnostic measurements are naturally correlated, such as radius, perimeter, and area.

In [ ]:
plt.figure(figsize=(12,8))
corr = X.corr()
sns.heatmap(corr, cmap='coolwarm', center=0, cbar=True)
plt.title('Feature Correlation Heatmap')
plt.show()

### Principal Component Analysis

PCA is used here for visualization to understand the underlying structure of the data in two dimensions.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
pca_df['diagnosis'] = df['diagnosis']

plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='diagnosis', alpha=0.8)
plt.title('PCA View of Diagnostic Measurements')
plt.show()

print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Total explained variance:', pca.explained_variance_ratio_.sum())

## 3. Data Preparation

Check for missing values and duplicate records. Then split the data into training and testing sets. Scaling is included inside model pipelines where needed.

In [ ]:
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Separate features and target
X = df.drop(columns=['diagnosis'])
y = data.target  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

## 4. Modeling

Train two models:
- Logistic Regression: interpretable and effective for linearly separable patterns.
- Random Forest: flexible ensemble model that can capture non-linear relationships.

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced'
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name} trained successfully.')

## 5. Evaluation

Evaluate each model using accuracy, precision, recall, F1-score, and ROC-AUC.

Because `sklearn` represents benign as class `1` and malignant as class `0`, this notebook calculates recall for malignant cases by setting `pos_label=0`.

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_malignant': precision_score(y_test, y_pred, pos_label=0),
        'recall_malignant': recall_score(y_test, y_pred, pos_label=0),
        'f1_malignant': f1_score(y_test, y_pred, pos_label=0),
        'roc_auc': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values(by='recall_malignant', ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

print('Best model selected based on malignant recall:', best_model_name)

y_pred_best = best_model.predict(X_test)
print('
Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=data.target_names))

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.title(f'Confusion Matrix: {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f'ROC Curve: {best_model_name}')
plt.show()

### What the Measures Tell Us

- **Accuracy** shows the overall percentage of correct predictions.
- **Precision for malignant cases** tells us how many cases predicted as malignant were actually malignant.
- **Recall for malignant cases** tells us how many actual malignant cases were correctly identified.
- **F1-score** balances precision and recall.
- **ROC-AUC** summarizes how well the model separates benign and malignant cases across thresholds.

For this project, recall for malignant cases is prioritized because the cost of missing a malignant case can be high in a screening-style context.

## 6. Prediction Scenario

Imagine a diagnostic support team receives a new case with measurements similar to one patient record. The model can estimate whether the case is likely to be malignant or benign.

Below, we use one example from the test set as a realistic scenario.

In [ ]:
scenario = X_test.iloc[[0]]
actual_label = 'malignant' if y_test[0] == 0 else 'benign'

prediction = best_model.predict(scenario)[0]
probability = best_model.predict_proba(scenario)[0]

predicted_label = 'malignant' if prediction == 0 else 'benign'

print('Scenario measurements:')
display(scenario.T.rename(columns={scenario.index[0]: 'value'}).head(15))

print('
Predicted diagnosis:', predicted_label)
print('Probability malignant:', round(probability[0], 4))
print('Probability benign:', round(probability[1], 4))
print('Actual label for this held-out case:', actual_label)

### Prediction Interpretation

The model returns both a predicted diagnosis and class probabilities. If the probability for malignant is high, the case should be prioritized for further expert review. If the probability for benign is high, the model is indicating that the measurement pattern resembles benign examples from the training data.

This result should be treated as decision support, not a final medical conclusion.

## 7. Conclusion

This project demonstrates a complete CRISP-DM machine learning workflow:

- Explored feature distributions, skewness, correlations, and PCA structure.
- Checked whether the data required cleaning.
- Trained and compared multiple classification models.
- Evaluated model performance using accuracy, recall, precision, F1-score, and ROC-AUC.
- Ran a prediction scenario and explained the result in plain language.

The same project structure can be adapted to other datasets and business problems.